<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [26]</a>'.</span>

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

# Constants

In [4]:
TAWJEEH_DATASET_NAME = 'xlsum'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/xlsum_arabic_experimental'
TASK_NAME='summarization'
MODEL_PATH = "/raid_storage/shared_models/Qwen3-8B-Base"
MODEL_NAME = "Qwen3-8B"

In [5]:
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [6]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14901,
  'tags': [],
  'name': 'A Simple Test Prompt',
  'task': {'name': 'dialect identification'},
  'status': 'DRAFT',
  'template': 'Please predict the most suitable dialect for the following text: {{arabic}}\xa0\r\n|||{{answer_choices[label]}}',
  'created_by': 'irfan',
  'dataset_name': 'arbml/AraBench_dev',
  'dataset_subset': 'default',
  'answer_choices': ['Tunisian',
   'MSA',
   'Morrocan',
   'Qatari',
   'Egyptian',
   'Lebanese'],
  'text_direction': 'ltr'},
 {'id': 14898,
  'tags': ['', 'Zero-shot COT'],
  'name': 'Prompt with zero-shot chain of thoughts',
  'task': {'name': 'claim verification'},
  'status': 'APPROVED',
  'template': "For the following task you have to label if the two sentences are of on of the following labels: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}. Sentence 1: {{s1}}\xa0 and sentence 2: {{s2}}.\r\nLet's think step by step:\r\n|||\r\n{{answer_choices[label]}}",
  'created_by': 'ahmed',


In [7]:
len(prompts)

365

In [8]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

352

## Finetuning

### Get the dataset prompts

In [9]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

7

In [10]:
SELECTED_PROMPTS_IDS = [
    14871,
    14803,
    14856,
    14858,
    14668,
]

In [11]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [12]:
import datasets

In [13]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['gem_id', 'url', 'title', 'target', 'references', 'text'],
        num_rows: 4689
    })
})

### Merge the prompts

In [14]:
from jinja2 import Environment, StrictUndefined

In [15]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    prefix = prefix.replace('\xa0','')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}|||{suffix.strip()}'

In [16]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e

### Perform prompt-merge on one example prompt, for experimentation

In [17]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][-700]))

You are an Arabic summarization expert! The summarization of the following Arabic passage: "(أرشيف) قالت السلطات المصرية إن الجيش والشرطة اللذين يشنان حملة على الجماعة قتلا مئات من أعضائها.

ونشرت الجماعة على موقع تويتر مقطعا مصورا يظهر عشرات القتلى من الضباط والجنود في الهجوم الذي وقع يوم 24 أكتوبر/ تشرين الأول مستهدفا نقطة التفتيش العسكرية في منطقة كرم القواديس قرب مدينة الشيخ زويد بشمال سيناء.

وظهر في الشريط المصور رجل يحذر الرئيس المصري عبد الفتاح السيسي من هجمات سوف تشنها الجماعة على قوات الجيش والشرطة في شمال سيناء.

وكتب في شريط في أسفل الشاشة "الاستشهادي أبو حمزة الأنصاري تقبله الله الغائر على نقطة كرم القواديس العسكرية."

لكن الشريط المصور لم يتضمن تاريخ الهجوم على النقطة ولم يتسن التحقق من صحته حتى الآن.

مواضيع قد تهمك نهاية

وأطلقت الجماعة المتشددة على نفسها اسم ولاية سيناء المصرية بعد أن أعلنت الخميس مبايعة زعيم تنظيم الدولة الاسلامية أبو بكر البغدادي.

وكانت جماعة أنصار بيت المقدس، وهي أكبر جماعة إسلامية متشددة في مصر، أعلنت انضمامها إلى تنظيم الدولة الإسلامية الذى استول

In [18]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

6000.0

In [19]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending You are an Arabic summarization expert! The summarization of the following Arabic passage: "{{text}}" is:
|||
{{target}} sample index: 0


rending Generate a summary for the following Arabic document: {{text}}
|||
{{target}} sample index: 6000


rending A concise and informative summary that captures the main points of the Arabic article "{{text}}" while maintaining its original meaning is:
|||
{{target}} sample index: 12000


rending Create tldr for the following text: {{text}}
|||
{{target}} sample index: 18000


rending This article {{text}}  with the title {{title}}  can be summarized as:
|||
{{target}} sample index: 24000


30000

## Finetune the LLM

In [20]:
GLOBAL_SEED = 42

In [21]:
import random
random.seed(GLOBAL_SEED)

In [22]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Qwen3Initializer, LoRAConfigRepository
from sklearn.model_selection import train_test_split

🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py
🚨 Config not found for parakeet. You can manually add it to HARDCODED_CONFIG_FOR_MODELS in utils/auto_docstring.py


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/message_generator.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [23]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Qwen3Initializer(),
)
llm_loader

In [24]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


`torch_dtype` is deprecated! Use `dtype` instead!


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

loading weights file /raid_storage/shared_models/Qwen3-8B-Base/model.safetensors.index.json


Instantiating Qwen3ForCausalLM model under default dtype torch.bfloat16.


...

loading file tokenizer_config.json


loading file chat_template.jinja


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/generation_config.json


Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}



In [25]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    prefix,suffix = sample.split('|||')
    prefix = prefix.strip()
    prefix = prefix.replace('\xa0','')
    suffix = suffix.strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('You are an Arabic summarization expert! The summarization of the following Arabic passage: "زعماء الترويكا الحاكمة في تونس يحتفلون بإقرار المجلس التأسيسي أول دستور بعد الثورة\n\n واوضح التصويت على الدستور الجديد وجود درجة عالية من التوافق على بنوده، اذ وافق عليه 200 عضو من اعضاء المجلس التأسيسي، والذين يبلغ عددهم 216 عضوا. وبشكل عام ينظر الى اقرار الدستور على انه خطوة سياسية هامة على طريق تحقيق الاستقرار وبناء المؤسسات في تونس. وتزامن مع اقرار الدستور الاتفاق على تشكيل حكومة جديدة من شخصيات مستقلة برئاسة مهدي جمعة. وكان الرئيس التونسي المنصف المرزوقي وصف تجربة بلاده "بالمعجزة التونسية"، اذ تمكنت تونس "من الحفاظ على الحرية والامن ونموذج من الاعتدال" على حد وصفه.\n\n غير ان هناك تحديات هائلة لازالت تواجه تونس، وتتطلب عملا شاقا للتعامل معها وتجاوزها. فعلى المستوى الاقتصادي يعاني الشباب التونسي من بطالة واسعة، ويعاني المجتمع كله من ارتفاع كبير في الاسعار، وهناك ضغوط لمواجهة العجز في موازنة الدولة، والتي ترجع في جزء اساسي منها الى الدعم الحكومي لمجموعة من السلع الغذائية و

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [26]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=1,
    eval_batch_size=1,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}',
    early_stopping_patience=20,
    eval_steps=1000,
)

PyTorch: setting up devices


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit/llm/llm_trainer.py:83: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


Using auto half precision backend



***** Running Evaluation *****


...

  Batch size = 1


loading configuration file /raid_storage/shared_models/Qwen3-8B-Base/config.json


Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 12288,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_at

{'eval_loss': 1.2974910736083984, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 350.3378, 'eval_samples_per_second': 8.563, 'eval_steps_per_second': 8.563, 'epoch': 0.2962962962962963}


OutOfMemoryError: CUDA out of memory. Tried to allocate 10.18 GiB. GPU 1 has a total capacity of 79.25 GiB of which 5.12 GiB is free. Including non-PyTorch memory, this process has 74.11 GiB memory in use. Of the allocated memory 71.71 GiB is allocated by PyTorch, and 1.90 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
exit()